### Data Ingestion

In [1]:
### document datastructure
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text content I am using to create my first ever RAG",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Nitish",
        "date_created": "2026-05-31"
    }
)

In [3]:
## Create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
# basic key-value pairs
sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction

    Python is a high-level, interpreted programming language known for its simplicity and readability.
    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
    programming languages in the world.

    Key Features:
    - Easy to learn and use
    - Extensive standard library
    - Cross-platform compatibility
    - Strong community support

    Python is widely used in web development, data science, artificial intelligence, and automation.""",
        "../data/text_files/machine_learning.txt": """Machine Learning Basics

    Machine learning is a subset of artificial intelligence that enables systems to learn and improve
    from experience without being explicitly programmed. It focuses on developing computer programs
    that can access data and use it to learn for themselves.

    Types of Machine Learning:
    1. Supervised Learning: Learning with labeled data
    2. Unsupervised Learning: Finding patterns in unlabeled data
    3. Reinforcement Learning: Learning through rewards and penalties

    Applications include image recognition, speech processing, and recommendation systems
    """,
}

# w ovverrrides here
for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [5]:
# from langchain.document_loaders import TextLoader #in new version
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document=loader.load()
print(document)

/tmp/ipykernel_14867/1705145242.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/home/paper/Desktop/nitis/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability.\n    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular\n    programming languages in the world.\n\n    Key Features:\n    - Easy to learn and use\n    - Extensive standard library\n    - Cross-platform compatibility\n    - Strong community support\n\n    Python is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn and improve\n    from experience without being explicitly programmed. It focuses on developing computer programs\n    that can access data and use it to learn for themselves.\n\n    Types of Machine Learning:\n    1. Supervised Learning: Learning with labeled data\n    2. Unsupervised Learning: Finding patterns in unlabeled data\n    3. Reinforcement Learning: Learning through rewards and penalties\n\n    Applications include image recognition, speech processing, and recommendation systems\n    '),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability.\n    Created by Guido van Rossum and first released in 1991, Python h

In [7]:
### Directory Loader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
pdf_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files
    loader_cls=PyMuPDFLoader, ##loader class to use
    show_progress=False
)

pdf_documents = pdf_loader.load()
pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_path': '../data/pdf/NIPS-2017-attention-is-all-you-need-Paper.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz

In [8]:
type (pdf_documents[0]) ## Document data type

langchain_core.documents.base.Document

### Embedding and vectorStoreDB

In [9]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
        """Handles document embedding generation using SentenceTransformer"""

        def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
            """
            Initialize the embedding manager

            Args:
                model_name: HuggingFace model name for sentence embeddings
            """
            self.model_name = model_name
            self.model = None
            self._load_model()
        
        def _load_model(self):
            """Load the SentenceTransformer model"""
            try:
                print(f"Loading embedding model: {self.model_name}")
                self.model = SentenceTransformer(self.model_name)
                print(f"Model loaded successfully. Embedding dimension: {self.get_embedding_dimension()}")
            except Exception as e:
                print(f"Error loading model {self.model_name}: {e}")
                raise

        def generate_embeddings(self, texts: List[str]) -> np.ndarray:
            """
            Generate embeddings for a list of texts

            Args:
                texts: List of text strings to embed

            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)
            """
            if not self.model:
                raise ValueError("Model is not loaded")

            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings

        def get_embedding_dimension(self) -> int:
            """"Get the embedding dimension of the model"""
            if not self.model:
                raise ValueError("Model not loaded")
            return self.model.get_sentence_embedding_dimension()

## initialise the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3581.40it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_14867/3016040027.py:47: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()
